# Actividad 11 – Clase 02

## Películas más similares a **"Star Trek Beyond"** basándonos en sus géneros

### Objetivo
A partir del archivo `tmdb_5000_movies.csv`, encontrar las **3 películas más similares** a
*Star Trek Beyond* usando la **similitud de Jaccard** aplicada al conjunto de sus géneros.

### ¿Qué es la distancia de Jaccard?
Si representamos cada película como el **conjunto de sus géneros**, podemos medir qué tan
parecidas son dos películas comparando esos conjuntos:

- **Intersección** (A ∩ B): géneros que tienen **ambas** películas.
- **Unión** (A ∪ B): todos los géneros que tiene **al menos una** de las dos.

El **índice de Jaccard** (similitud) es:

$$J(A,B) = \frac{|A \cap B|}{|A \cup B|}$$

y la **distancia de Jaccard** es su complemento:

$$d_{Jaccard}(A,B) = 1 - J(A,B) = \frac{|A \cup B| - |A \cap B|}{|A \cup B|}$$

Interpretación:
- **0.0** → los conjuntos son idénticos (máxima similitud).
- **1.0** → no comparten ningún género (totalmente distintos).

Ejemplo: *Star Trek Beyond* = {Action, Adventure, Science Fiction} y *Avatar* =
{Action, Adventure, Fantasy, Science Fiction} → intersección = 3, unión = 4 →
distancia = 1 − 3/4 = **0.25**.

### Plan de trabajo
1. Cargar el dataset con `pandas`.
2. Parsear la columna `genres` (es una lista en formato JSON) y convertir cada película en un `set` de géneros.
3. Definir la función de distancia de Jaccard.
4. Calcular la distancia entre *Star Trek Beyond* y **todas** las demás películas.
5. Ordenar y seleccionar las 3 con menor distancia (más similares).
6. Mostrar la **tabla completa** de distancias.

In [1]:
import json
import pandas as pd

df = pd.read_csv("tmdb_5000_movies.csv")
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])
df.columns.tolist()

Filas: 4803 | Columnas: 20


['budget',
 'genres',
 'homepage',
 'id',
 'keywords',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'status',
 'tagline',
 'title',
 'vote_average',
 'vote_count']

### Paso 1: Preparar los conjuntos de géneros

La columna `genres` contiene texto que parece una lista de diccionarios en formato JSON,
por ejemplo:

```
[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]
```

En este bloque:
- Definimos una función que parsea ese texto con `json.loads` y extrae solo los `name`
  de cada género, devolviendo un **`set`** (conjunto) de géneros.
- Si la celda está vacía o el formato no se puede leer, devolvemos un conjunto vacío.
- Guardamos ese conjunto en una nueva columna `generos_conjunto`.
- Verificamos que *Star Trek Beyond* exista y mostramos sus géneros.

In [2]:
def extraer_generos(texto):
    try:
        lista = json.loads(texto)
        return {g["name"] for g in lista}
    except Exception:
        return set()

df["generos_conjunto"] = df["genres"].apply(extraer_generos)

objetivo = df[df["original_title"] == "Star Trek Beyond"].iloc[0]
print("Película objetivo:", objetivo["original_title"])
print("Géneros:", sorted(objetivo["generos_conjunto"]))

Película objetivo: Star Trek Beyond
Géneros: ['Action', 'Adventure', 'Science Fiction']


### Paso 2: Definir la distancia de Jaccard

Implementamos la fórmula vista anteriormente:

$$d_{Jaccard}(A,B) = 1 - \frac{|A \cap B|}{|A \cup B|}$$

Consideración especial: si la unión es vacía (0/0), definimos la distancia como **1.0**,
es decir, máxima diferencia, porque no hay géneros con qué comparar.

In [3]:
def distancia_jaccard(a, b):
    union = a | b
    if not union:
        return 1.0
    return 1 - len(a & b) / len(union)

### Paso 3: Calcular la distancia contra todas las películas

Para cada fila del dataset calculamos la distancia de Jaccard entre su conjunto de
géneros y el conjunto de géneros de *Star Trek Beyond*. Este paso es **vectorizado**:
aplicamos la función a toda la columna de conjuntos usando `apply`.

Además creamos una columna extra con los géneros como texto legible (`, ` separa cada
género) para que la tabla final sea cómoda de leer. Excluimos después a la propia
película objetivo (su distancia con ella misma siempre sería 0 y no debe contar).

In [4]:
resultados = df[["original_title", "title", "generos_conjunto"]].copy()
resultados["distancia_jaccard"] = resultados["generos_conjunto"].apply(
    lambda g: distancia_jaccard(objetivo["generos_conjunto"], g)
)
resultados["generos_texto"] = resultados["generos_conjunto"].apply(
    lambda g: ", ".join(sorted(g))
)
resultados = resultados[resultados["original_title"] != "Star Trek Beyond"].reset_index(drop=True)
print("Películas comparadas:", len(resultados))

Películas comparadas: 4802


### Paso 4: Las 3 más similares y la tabla completa

La distancia de Jaccard es una **medida de diferencia**: cuanto más pequeña, más
similares son las películas. Por eso:

1. Ordenamos el DataFrame por `distancia_jaccard` de menor a mayor (**ascendente**).
2. Las 3 primeras filas son las **3 películas más similares** en género.
3. También se imprime la **tabla completa** de distancias para poder revisar todas las
   películas y ver la similitud de cada una.

Nota: pueden existir empates con distancia **0.0** si varias películas comparten
exactamente los mismos tres géneros; en ese caso se muestran las primeras 3 según el
orden del dataset.

In [5]:
similares = resultados.sort_values("distancia_jaccard")
top3 = similares.head(3)

print("=" * 70)
print("TOP 3 PELÍCULAS MÁS SIMILARES A 'STAR TREK BEYOND'")
print("=" * 70)
print(top3[["original_title", "generos_texto", "distancia_jaccard"]].to_string(index=True))
print()
print("TABLA COMPLETA (todas las películas, ordenadas por distancia)")
print("=" * 70)
print(similares[["original_title", "generos_texto", "distancia_jaccard"]].to_string())

TOP 3 PELÍCULAS MÁS SIMILARES A 'STAR TREK BEYOND'
          original_title                       generos_texto  distancia_jaccard
100   X-Men: First Class  Action, Adventure, Science Fiction                0.0
67              Iron Man  Action, Adventure, Science Fiction                0.0
1078                Dune  Action, Adventure, Science Fiction                0.0

TABLA COMPLETA (todas las películas, ordenadas por distancia)
                                                                              original_title                                                           generos_texto  distancia_jaccard
100                                                                       X-Men: First Class                                      Action, Adventure, Science Fiction           0.000000
67                                                                                  Iron Man                                      Action, Adventure, Science Fiction           0.000000
1078          